# Resultados iniciales
## Tema 13: Identificación de menciones de entidades biomédicas en resúmenes de investigación

Integrantes:
- José Ricardo Méndez González, 21289
- Sara María Pérez Echeverría, 21371
- Emily Elvia Melissa Pérez Alarcón, 21385
- Adrian Fulladolsa Palma, 21592

In [33]:
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
import pandas as pd
import nltk
import re

### Preprocesamiento

In [34]:
# download necessary nltk resources
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\PiCi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\PiCi\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [35]:
# function to clean and preprocess text
def preprocessText(text):
    # convert to lowercase
    text = text.lower()
    
    # remove non-alphabetic characters
    text = re.sub(r'[^a-z\s]', '', text)
    
    # tokenize by spaces
    tokens = text.split()
    
    # remove stopwords
    stopWords = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stopWords]
    
    # lemmatization
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return ' '.join(tokens)

In [36]:
# path of the data
dataPath = '../data/'

# load data
abstractsTrain = pd.read_csv(dataPath + 'abstracts_train.csv', on_bad_lines='skip', delimiter='\t')
entitiesTrain = pd.read_csv(dataPath + 'entities_train.csv', on_bad_lines='skip', delimiter='\t')
relationsTrain = pd.read_csv(dataPath + 'relations_train.csv', on_bad_lines='skip', delimiter='\t')
abstractsTest = pd.read_csv(dataPath + 'abstracts_test.csv', on_bad_lines='skip', delimiter='\t')

In [37]:
# apply the preprocessing function to the abstract text column
abstractsTrain['cleanAbstract'] = abstractsTrain['abstract'].apply(preprocessText)
abstractsTest['cleanAbstract'] = abstractsTest['abstract'].apply(preprocessText)

# show the cleaned texts
print(abstractsTrain[['abstract', 'cleanAbstract']].head())
print(abstractsTest[['abstract', 'cleanAbstract']].head())

                                            abstract  \
0  We report on a new allele at the arylsulfatase...   
1  Classical phenylketonuria is an autosomal rece...   
2  The metabolism of the cardioselective beta-blo...   
3  Previous experiments in this laboratory have s...   
4  Eighty unrelated individuals with Duchenne mus...   

                                       cleanAbstract  
0  report new allele arylsulfatase arsa locus cau...  
1  classical phenylketonuria autosomal recessive ...  
2  metabolism cardioselective betablocker metopro...  
3  previous experiment laboratory shown microinje...  
4  eighty unrelated individual duchenne muscular ...  
                                            abstract  \
0  The effect of induced hypertension instituted ...   
1  A linkage study in 30 Becker muscular dystroph...   
2  The effects of a 6-hour infusion with haloperi...   
3  Fragments of the adrenoleukodystrophy (ALD) cD...   
4  The ability to scan a large gene rapidly and a... 

In [38]:
# group entities by abstract_id
entities_grouped = entitiesTrain.groupby('abstract_id')['type'].apply(list).reset_index()

# merge the cleaned abstracts with their corresponding entities by 'abstract_id'
data_merged = pd.merge(abstractsTrain, entities_grouped, how='inner', on='abstract_id')

### Implementación de modelos

In [39]:
from sklearn.model_selection import train_test_split

# dataset splitting, 80% for training and 20% for testing
X_train, X_val, y_train, y_val = train_test_split(
    data_merged['cleanAbstract'], data_merged['type'], test_size=0.2, random_state=42,
)

# sizes of the splits
print(f'Training set size: {len(X_train)}')
print(f'Validation set size: {len(X_val)}')

Training set size: 320
Validation set size: 80


#### Long Short-Term Memory Networks (LSTM)

In [40]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from sklearn.preprocessing import MultiLabelBinarizer

tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)

max_sequence_length = max([len(seq) for seq in X_val_seq])

mlb = MultiLabelBinarizer()
y_train_encoded = mlb.fit_transform(y_train)
y_val_encoded = mlb.transform(y_val)

max_length = max(len(seq) for seq in X_train_seq)
X_train_pad = pad_sequences(X_train_seq, maxlen=max_length, padding='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=max_length, padding='post')

model = Sequential([
    Embedding(input_dim=len(tokenizer.word_index) + 1, output_dim=128, input_length=max_length),
    LSTM(units=128, return_sequences=False),
    Dense(units=len(mlb.classes_), activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.fit(X_train_pad, y_train_encoded, validation_data=(X_val_pad, y_val_encoded), epochs=30, batch_size=32)

Epoch 1/30


c:\Users\PiCi\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 107ms/step - accuracy: 0.2703 - loss: 0.6275 - val_accuracy: 0.2750 - val_loss: 0.4477
Epoch 2/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 84ms/step - accuracy: 0.3650 - loss: 0.4481 - val_accuracy: 0.2750 - val_loss: 0.4547
Epoch 3/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 80ms/step - accuracy: 0.3650 - loss: 0.4384 - val_accuracy: 0.2750 - val_loss: 0.4503
Epoch 4/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 80ms/step - accuracy: 0.3650 - loss: 0.4416 - val_accuracy: 0.2750 - val_loss: 0.4491
Epoch 5/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 81ms/step - accuracy: 0.3650 - loss: 0.4374 - val_accuracy: 0.2750 - val_loss: 0.4478
Epoch 6/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 81ms/step - accuracy: 0.3650 - loss: 0.4377 - val_accuracy: 0.2750 - val_loss: 0.4477
Epoch 7/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 79ms/step - accuracy: 0.3650 - loss: 0.4383 - val_accuracy: 0.2750 - val_loss: 0.4482
Epoch 8/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 79ms/step - accuracy: 0.3650 - loss: 0.4379 - val_accuracy: 0.2750 - val_loss: 0

#### Support Vector Machines (SVM)

In [41]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report

mlb = MultiLabelBinarizer()
y_train_encoded = mlb.fit_transform(y_train)
y_val_encoded = mlb.transform(y_val)

vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)

svm_model = OneVsRestClassifier(SVC(kernel='linear', probability=True))

svm_model.fit(X_train_tfidf, y_train_encoded)

y_pred = svm_model.predict(X_val_tfidf)

print(classification_report(y_val_encoded, y_pred, target_names=mlb.classes_))


                            precision    recall  f1-score   support

                  CellLine       0.25      0.14      0.18         7
            ChemicalEntity       0.83      0.80      0.82        56
DiseaseOrPhenotypicFeature       0.94      1.00      0.97        75
         GeneOrGeneProduct       0.83      0.98      0.90        66
             OrganismTaxon       0.88      1.00      0.93        70
           SequenceVariant       0.89      0.89      0.89        35

                 micro avg       0.87      0.93      0.90       309
                 macro avg       0.77      0.80      0.78       309
              weighted avg       0.86      0.93      0.89       309
               samples avg       0.87      0.94      0.89       309



#### Graph Convolutional Networks (GCN)

In [42]:
import networkx as nx
import numpy as np

G = nx.Graph()

for idx, (abstract, entities) in enumerate(zip(X_train, y_train)):
    for entity in entities:
        unique_entity_id = f"{idx}_{entity}"
        
        G.add_node(unique_entity_id, abstract_id=idx, entity=abstract, x=np.random.rand(128))

for idx, entities in enumerate(y_train):
    for i in range(len(entities)):
        for j in range(i + 1, len(entities)):
            node_id_1 = f"{idx}_{entities[i]}"
            node_id_2 = f"{idx}_{entities[j]}"
            G.add_edge(node_id_1, node_id_2, relation_type="related")

standard_attributes = {'abstract_id': None, 'entity': None}
for node in G.nodes:
    for attr, default_value in standard_attributes.items():
        if attr not in G.nodes[node]:
            G.nodes[node][attr] = default_value


In [43]:
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.utils import from_networkx
from sklearn.preprocessing import MultiLabelBinarizer
import numpy as np

data = from_networkx(G)

data.x = torch.tensor([G.nodes[node]['x'] for node in G.nodes], dtype=torch.float32)

mlb = MultiLabelBinarizer()
y_encoded = mlb.fit_transform(y_train)  
y_full = np.zeros((data.num_nodes, len(mlb.classes_)))

for idx, entities in enumerate(y_train):
    y_encoded = mlb.transform([entities])[0]
    y_full[idx] = y_encoded

data.y = torch.tensor(y_full, dtype=torch.float32)

class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x 

modelGCN = GCN(in_channels=128, hidden_channels=16, out_channels=len(mlb.classes_))
optimizer = torch.optim.Adam(modelGCN.parameters(), lr=0.01, weight_decay=5e-4)

def train():
    modelGCN.train()
    optimizer.zero_grad()
    out = modelGCN(data.x, data.edge_index)
    loss = F.binary_cross_entropy_with_logits(out, data.y)
    loss.backward()
    optimizer.step()
    return loss.item()

for epoch in range(500):
    loss = train()
    print(f'Epoch {epoch}: Loss {loss:.4f}')


Epoch 0: Loss 0.6851
Epoch 1: Loss 0.5787
Epoch 2: Loss 0.5193
Epoch 3: Loss 0.4874
Epoch 4: Loss 0.4726
Epoch 5: Loss 0.4721
Epoch 6: Loss 0.4748
Epoch 7: Loss 0.4734
Epoch 8: Loss 0.4669
Epoch 9: Loss 0.4576
Epoch 10: Loss 0.4486
Epoch 11: Loss 0.4419
Epoch 12: Loss 0.4375
Epoch 13: Loss 0.4343
Epoch 14: Loss 0.4319
Epoch 15: Loss 0.4304
Epoch 16: Loss 0.4300
Epoch 17: Loss 0.4303
Epoch 18: Loss 0.4305
Epoch 19: Loss 0.4298
Epoch 20: Loss 0.4282
Epoch 21: Loss 0.4263
Epoch 22: Loss 0.4245
Epoch 23: Loss 0.4234
Epoch 24: Loss 0.4232
Epoch 25: Loss 0.4236
Epoch 26: Loss 0.4242
Epoch 27: Loss 0.4246
Epoch 28: Loss 0.4247
Epoch 29: Loss 0.4242
Epoch 30: Loss 0.4234
Epoch 31: Loss 0.4224
Epoch 32: Loss 0.4217
Epoch 33: Loss 0.4212
Epoch 34: Loss 0.4210
Epoch 35: Loss 0.4210
Epoch 36: Loss 0.4209
Epoch 37: Loss 0.4208
Epoch 38: Loss 0.4204
Epoch 39: Loss 0.4199
Epoch 40: Loss 0.4193
Epoch 41: Loss 0.4187
Epoch 42: Loss 0.4182
Epoch 43: Loss 0.4178
Epoch 44: Loss 0.4174
Epoch 45: Loss 0.417

#### Bidirectional Encoder Representations from Transformers (BERT)

In [44]:
# %pip install transformers

In [45]:
from transformers import BertTokenizer, BertForSequenceClassification
num_classes = 6
tokenizerBert = BertTokenizer.from_pretrained("bert-base-uncased")
modelBERT = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=num_classes)

mlb = MultiLabelBinarizer()
y_train_encoded = mlb.fit_transform(y_train)
y_val_encoded = mlb.transform(y_val)

def preprocess_texts_BERT(texts, labels):
    inputs = tokenizerBert(
        texts, padding=True, truncation=True, max_length=128, return_tensors="pt"
    )
    inputs["labels"] = torch.tensor(labels, dtype=torch.float)
    return inputs

train_texts = list(X_train)
train_labels = y_train_encoded
train_data = preprocess_texts_BERT(train_texts, train_labels)

val_texts = list(X_val)
val_labels = y_val_encoded
val_data = preprocess_texts_BERT(val_texts, val_labels)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [46]:
from torch.utils.data import Dataset

class TextDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        return {key: tensor[idx] for key, tensor in self.encodings.items()}

train_dataset = TextDataset(train_data)
val_dataset = TextDataset(val_data)

In [47]:
# %pip install transformers
# %pip install transformers[torch]
# %pip install accelerate

In [48]:
# import os

# os.environ["TRANSFORMERS_NO_TF"] = "1"
from transformers import Trainer, TrainingArguments


training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=modelBERT,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()


c:\Users\PiCi\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\training_args.py:1559: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
  8%|▊         | 10/120 [00:19<03:36,  1.96s/it]

{'loss': 0.6811, 'grad_norm': 2.7048163414001465, 'learning_rate': 1.0000000000000002e-06, 'epoch': 0.25}


 17%|█▋        | 20/120 [00:39<03:14,  1.95s/it]

{'loss': 0.671, 'grad_norm': 2.2681965827941895, 'learning_rate': 2.0000000000000003e-06, 'epoch': 0.5}


 25%|██▌       | 30/120 [00:59<02:58,  1.98s/it]

...

                                                 
100%|██████████| 120/120 [04:16<00:00,  1.95s/it]

{'eval_loss': 0.3930191397666931, 'eval_runtime': 5.1914, 'eval_samples_per_second': 15.41, 'eval_steps_per_second': 1.926, 'epoch': 3.0}


100%|██████████| 120/120 [04:17<00:00,  2.15s/it]

{'train_runtime': 257.4529, 'train_samples_per_second': 3.729, 'train_steps_per_second': 0.466, 'train_loss': 0.5454274415969849, 'epoch': 3.0}


TrainOutput(global_step=120, training_loss=0.5454274415969849, metrics={'train_runtime': 257.4529, 'train_samples_per_second': 3.729, 'train_steps_per_second': 0.466, 'total_flos': 63148921159680.0, 'train_loss': 0.5454274415969849, 'epoch': 3.0})

In [49]:
def preprocess_text_for_prediction(text):
    inputs = tokenizerBert(
        text, padding=True, truncation=True, max_length=128, return_tensors="pt"
    )
    return inputs["input_ids"], inputs["attention_mask"]

### Predicciones

In [50]:
import re

def extract_entity_ids(abstract):
    matched_entity_ids = []
    tokens = re.findall(r'\w+', abstract.lower())

    for node_id, node_data in G.nodes(data=True):
        entity_text = node_data.get('entity', '')
        if entity_text:
            entity_tokens = set(re.findall(r'\w+', entity_text.lower()))
            if entity_tokens.intersection(tokens):  
                matched_entity_ids.append(node_id)
    return matched_entity_ids

In [51]:
def predict_entities_lstm(abstract):
    sequence = tokenizer.texts_to_sequences([abstract])
    padded_sequence = pad_sequences(sequence, maxlen=max_sequence_length)
    prediction = model.predict(padded_sequence)
    predicted_entities = mlb.inverse_transform(prediction > 0.5)
    return predicted_entities[0]  

def predict_entities_svm(abstract):
    tfidf_sequence = vectorizer.transform([abstract])
    prediction = svm_model.predict(tfidf_sequence)
    predicted_entities = mlb.inverse_transform(prediction)
    return predicted_entities[0] 

def predict_entities_gcn(abstract):
    entity_ids = extract_entity_ids(abstract)
    subgraph_nx = G.subgraph(entity_ids)
    subgraph_data = from_networkx(subgraph_nx)
    if subgraph_data.x:
        subgraph_data.x = subgraph_data.x.float()
        
        modelGCN.eval()
        with torch.no_grad():
            out = modelGCN(subgraph_data.x, subgraph_data.edge_index)
        
        predicted_classes = (out > 0.5).cpu().numpy()
        predicted_entities = mlb.inverse_transform(predicted_classes)
        return predicted_entities[0]
    return ()

def predict_entities_bert(abstract):
    input_ids, attention_mask = preprocess_text_for_prediction(abstract)
    
    modelBERT.eval()
    with torch.no_grad():
        outputs = modelBERT(input_ids, attention_mask=attention_mask)
    predicted_classes = (outputs.logits > 0.5).cpu().numpy()
    predicted_entities = mlb.inverse_transform(predicted_classes)
    
    return predicted_entities[0] 

In [55]:
abstract_example = "metachromatic"
print("LSTM Prediction:", predict_entities_lstm(abstract_example))
print("SVM Prediction:", predict_entities_svm(abstract_example))
print("GCN Prediction:", predict_entities_gcn(abstract_example))
print("BERT Prediction:", predict_entities_bert(abstract_example))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
LSTM Prediction: ('ChemicalEntity', 'DiseaseOrPhenotypicFeature', 'GeneOrGeneProduct', 'OrganismTaxon')
SVM Prediction: ('ChemicalEntity', 'DiseaseOrPhenotypicFeature', 'GeneOrGeneProduct', 'OrganismTaxon')
GCN Prediction: ()
BERT Prediction: ('ChemicalEntity', 'DiseaseOrPhenotypicFeature', 'OrganismTaxon')
